# Coffee Standard J25 — D0DIRECT seed 42
Matched native control from official YOLO26n pretrained on the frozen 451-source split. Only train/validation are extracted; test is never extracted. Run the thesis-provenance audit notebook first.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import csv, importlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
ARM='D0DIRECT'; BRANCH='codex/coffee-standard-primary-audit'
REPO=Path('/content/coffee-bean-detection'); WORK=Path('/content')
os.chdir(WORK)
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96','gdown'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
import torch
if not torch.cuda.is_available(): raise RuntimeError('Aktifkan GPU Colab')
from coffee_detector.drive_project import resolve_drive_project_root
from coffee_detector.analysis.coffee_standard_j25_thesis_provenance import audit_j25_thesis_provenance
from coffee_detector.data.prepare_coffee_standard_j25_source_split import prepare_j25_source_split
PROJECT=resolve_drive_project_root()
ARCHIVE=WORK/'data_aug_11.zip'
if not ARCHIVE.is_file(): subprocess.run([sys.executable,'-m','gdown','https://drive.google.com/uc?id=1AofT7VbiNFM8ul-0vyCAKj7Rp4j5OX0f','-O',str(ARCHIVE)],check=True)
PROVENANCE=WORK/'coffee_standard_j25_thesis_provenance.json'
provenance=audit_j25_thesis_provenance(ARCHIVE,PROVENANCE)
if not provenance['decision'].startswith('PASS'): raise RuntimeError(f'Provenance gagal: {provenance["decision"]}')
PROVENANCE_EVIDENCE=PROJECT/'evidence/coffee-standard-j25-thesis-provenance-v1/coffee_standard_j25_thesis_provenance.json'; PROVENANCE_EVIDENCE.parent.mkdir(parents=True,exist_ok=True)
if not PROVENANCE_EVIDENCE.is_file(): shutil.copy2(PROVENANCE,PROVENANCE_EVIDENCE)
else:
    old=json.loads(PROVENANCE_EVIDENCE.read_text()); assert old.get('source_archive_sha256')==provenance['source_archive_sha256'] and str(old.get('decision','')).startswith('PASS'), 'Evidence provenance Drive berbeda'
DATA=WORK/'coffee-standard-j25-source-split'
if DATA.exists(): shutil.rmtree(DATA)
contract=prepare_j25_source_split(ARCHIVE,DATA,seed=42)
EVIDENCE=PROJECT/'evidence/coffee-standard-j25-source-split-v1'; EVIDENCE.mkdir(parents=True,exist_ok=True)
for name in ('coffee_standard_j25_source_split_summary.json','coffee_standard_j25_source_split_manifest.json'):
    source=DATA/name; target=EVIDENCE/name
    if target.is_file():
        old=json.loads(target.read_text())
        if name.endswith('_summary.json'): assert isinstance(old,dict) and old.get('source_archive_sha256')==contract['source_archive_sha256'], f'Evidence summary Drive berbeda: {target}'
        else: assert isinstance(old,list) and len(old)==contract['source_identities'], f'Evidence manifest Drive berbeda: {target}'
    if not target.is_file(): shutil.copy2(source,target)
CONTRACT=DATA/'coffee_standard_j25_source_split_summary.json'
from ultralytics import YOLO
_=YOLO('yolo26n.pt'); PRETRAINED=REPO/'yolo26n.pt'
OUT=PROJECT/'experiments/coffee-standard-j25-af2-direct-v1'; OUT.mkdir(parents=True,exist_ok=True)
print('ARM:',ARM,'GPU:',torch.cuda.get_device_name(0),'DATA:',contract['images'],'OUT:',OUT)


In [ ]:
LOG=OUT/f'{ARM}_seed42_run.log'
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_coffee_standard_j25_af2_direct','--arm',ARM,'--data-root',str(DATA),'--development-contract',str(CONTRACT),'--provenance-summary',str(PROVENANCE),'--pretrained-checkpoint',str(PRETRAINED),'--output-root',str(OUT),'--seed','42','--device','0','--authorize-training']
print('START/RESUME:',ARM,'| log=',LOG,flush=True)
with LOG.open('a',encoding='utf-8') as stream: process=subprocess.Popen(command,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT)
last=-1
while process.poll() is None:
    results=OUT/ARM/f'{ARM}_seed42'/'results.csv'
    epochs=sum(1 for _ in csv.DictReader(results.open(encoding='utf-8'))) if results.is_file() else 0
    if epochs!=last: print(f'{ARM}: {epochs}/50 epoch tercatat',flush=True); last=epochs
    time.sleep(120)
if process.returncode:
    print('\n'.join(LOG.read_text(errors='replace').splitlines()[-120:])); raise RuntimeError(f'{ARM} gagal: {process.returncode}')
RESULT=OUT/'val_reports'/f'{ARM}_seed42_result.json'
print(json.dumps(json.loads(RESULT.read_text()),indent=2,ensure_ascii=False))
print('last.pt tersimpan di Drive setiap epoch. Test tidak diekstrak.')
